# Day 5 — Pipeline Automation & Data Quality

## Overview
This notebook demonstrates the complete automated pipeline with:
- **Daily incremental updates** (fetches missing days from latest database date to current date)
- **Automated quality checks** with pass/fail reporting
- **Comprehensive logging** for traceability
- **Full and incremental modes**
- **94 cities support** with precise coordinates

## Key Features Implemented

1. **Daily Update Logic**: Pipeline fetches only missing days between max database date and current date
2. **Quality Gates**: Automated checks after each pipeline stage
3. **Incremental Loading**: Fetches only new data since last update
4. **Force Update Option**: Override daily rule when needed
5. **City-Specific Updates**: Targeted updates for selected cities

### Pipeline Architecture
```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   Ingestion    │ →  │   Database     │ →  │   Cleaning     │
│ (API calls)    │    │  (DuckDB)      │    │  (Staging)     │
└─────────────────┘    └─────────────────┘    └─────────────────┘
         ↓                      ↓                      ↓
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│ Quality Checks │    │ Quality Checks │    │ Quality Checks │
│   (Raw)       │    │  (Staging)    │    │ (Analytics)    │
└─────────────────┘    └─────────────────┘    └─────────────────┘
         ↓                      ↓                      ↓
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   Features     │ →  │   Analytics    │ →  │   Reports     │
│ Engineering    │    │   Tables       │    │   & Logs      │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

### Daily Update Logic
The pipeline implements smart daily updates:
- ✅ **Data Freshness Check**: Checks if there are missing days between latest date and current date
- ✅ **Skip Logic**: Skips update if no missing days (saves API calls)
- ✅ **Force Override**: `--force-update` flag bypasses freshness rule
- ✅ **City-by-City**: Each city's data freshness is checked individually

## 0  Imports & path setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure the src/ folder is importable
PROJECT_ROOT = Path.cwd().parent   # adjust if you run from a different folder
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Paths used throughout
DB_PATH   = PROJECT_ROOT / 'data' / 'weather.duckdb'
DATA_DIR  = PROJECT_ROOT / 'data'
LOG_DIR   = PROJECT_ROOT / 'logs'

print('Project root:', PROJECT_ROOT)
print('DB path     :', DB_PATH)


## 3  Weekly Incremental Update

Now let's test the **weekly incremental update** feature. The pipeline should:
- Check if data is 7+ days old
- Skip update if data is fresh  
- Only fetch new data if needed
- Respect the weekly update rule

Equivalent to:
```bash
python src/pipeline.py --mode incremental
```

Because we just ran a full load, pipeline should detect that all cities are
already up-to-date and skip the API fetch gracefully.

In [ ]:
summary_inc = run_pipeline(
    mode     = 'incremental',
    db_path  = DB_PATH,
    data_dir = DATA_DIR,
    log_dir  = LOG_DIR,
)

print('\n── Incremental run summary ───────────────')
for k, v in summary_inc.items():
    print(f'  {k:<20}: {v}')

print(f'\nCities skipped (already up-to-date): {summary_inc.get("cities_skipped", 0)}')
print(f'New rows ingested: {summary_inc.get("rows_ingested", 0)}')

# Check if weekly update logic worked
if summary_inc.get("cities_skipped", 0) > 0:
    print("✅ Weekly update logic working - skipped fresh data")
else:
    print("ℹ️  All cities needed updates or force update was used")

## 5  Force Update Test

Let's test the **force update** option that overrides the weekly rule. This is useful when:
- You need to update data regardless of the 7-day rule
- Testing the pipeline
- Manual data refresh is needed

Equivalent to:
```bash
python src/pipeline.py --mode incremental --force-update
```

This will override the 7-day rule and update data regardless of freshness.

In [ ]:
# Test force update (overrides weekly rule)
summary_force = run_pipeline(
    mode       = 'incremental',
    db_path    = DB_PATH,
    data_dir   = DATA_DIR,
    log_dir    = LOG_DIR,
    force_update = True,  # This overrides the 7-day rule
)

print('\n── Force update summary ─────────────────')
for k, v in summary_force.items():
    print(f'  {k:<20}: {v}')

print(f'\nForce update used: {summary_force.get("force_update", False)}')
print(f'New rows ingested: {summary_force.get("rows_ingested", 0)}')

if summary_force.get("rows_ingested", 0) > 0:
    print("✅ Force update successful - new data fetched")
else:
    print("ℹ️  No new data available or API limits reached")

## 3  Incremental pipeline run

Equivalent to:
```bash
python src/pipeline.py --mode incremental
```

Because we just ran a full load, the pipeline should detect that all cities are
already up-to-date and skip the API fetch gracefully.


In [ ]:
summary_inc = run_pipeline(
    mode     = 'incremental',
    db_path  = DB_PATH,
    data_dir = DATA_DIR,
    log_dir  = LOG_DIR,
)

print('\n── Incremental run summary ───────────────')
for k, v in summary_inc.items():
    print(f'  {k:<20}: {v}')

print(f'\nCities skipped (already up-to-date): {summary_inc.get("cities_skipped", 0)}')
print(f'New rows ingested: {summary_inc.get("rows_ingested", 0)}')


## 4  Quality gate results (standalone demo)

In [ ]:
import duckdb
from quality_checks import run_all_checks, print_quality_report

conn = duckdb.connect(str(DB_PATH))

try:
    raw_df      = conn.execute('SELECT * FROM raw_historical').df()
except Exception:
    raw_df = None

try:
    staging_df  = conn.execute('SELECT * FROM staging_historical').df()
except Exception:
    staging_df = None

try:
    analytics_df = conn.execute('SELECT * FROM analytics_historical').df()
except Exception:
    analytics_df = None

conn.close()

results = run_all_checks(
    raw_df       = raw_df,
    staging_df   = staging_df,
    analytics_df = analytics_df,
)
print_quality_report(results)

# Display as a DataFrame for notebook clarity
pd.DataFrame(results)


## 5  View pipeline log

In [ ]:
log_file = LOG_DIR / 'pipeline.log'

if log_file.exists():
    lines = log_file.read_text(encoding='utf-8').splitlines()
    # Show last 40 lines
    print(f'... (showing last 40 of {len(lines)} lines) ...\n')
    print('\n'.join(lines[-40:]))
else:
    print(f'Log file not found at {log_file}')


## 6  Pipeline Architecture Diagram

```
╔══════════════════════════════════════════════════════════════════════════╗
║             HEAT-PULSE  —  End-to-End Pipeline Architecture              ║
╚══════════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────────────────────────────────────────────┐
  │                        TRIGGER (CLI)                                │
  │   python src/pipeline.py --mode [full | incremental]                │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 1 — INGEST            (src/ingestion.py + pipeline.py)       │
  │                                                                     │
  │  full mode       → fetch ALL dates from Open-Meteo API              │
  │  incremental     → check max(time) per city in raw_historical        │
  │                    fetch only dates AFTER that                      │
  │                    APPEND (not replace) new rows                    │
  │                                                                     │
  │  Output → DuckDB: raw_historical table                              │
  │         → data/raw.parquet  (file-system backup)                    │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #1           │
                    │  ✓ row_count  (FAIL→abort) │
                    │  ✓ freshness  (WARN)       │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 2 — CLEAN             (src/cleaning.py)                      │
  │                                                                     │
  │  handle_missing_values()  → ffill temp, zero precip, interpolate    │
  │  flag_outliers()          → IQR flags, rows kept                    │
  │  validate_date_continuity()→ gap audit per city                     │
  │                                                                     │
  │  Output → DuckDB: staging_historical table                          │
  │         → data/staging_historical.parquet                           │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #2           │
                    │  ✓ null_ratio    (WARN)    │
                    │  ✓ date_contin.  (WARN)    │
                    │  ✓ value_ranges  (WARN)    │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 3 — FEATURES          (src/features.py)                      │
  │                                                                     │
  │  add_rolling_averages()   → 7d / 30d means                          │
  │  add_seasonal_indicators()→ month, quarter, season                  │
  │  add_temperature_range()  → max − min                               │
  │  add_degree_days()        → HDD / CDD @ 18°C base                  │
  │  add_anomaly_score()      → deviation from DOY mean                 │
  │  add_lag_features()       → lag-1 / lag-2 temp & precip             │
  │                                                                     │
  │  Output → DuckDB: analytics_historical table                        │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #3           │
                    │  ✓ feature_completeness    │
                    │    (WARN if cols missing)  │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  LOGGING  (logs/pipeline.log)                                       │
  │  • Start / end time & total duration                                │
  │  • Rows per stage per city                                          │
  │  • Errors & warnings                                                │
  │  • All quality gate results                                         │
  └─────────────────────────────────────────────────────────────────────┘

  Data layers in DuckDB
  ─────────────────────
  raw_historical       ← untouched API responses
  staging_historical   ← cleaned, nulls handled, outliers flagged
  analytics_historical ← ML-ready features added
```


## 7  CLI equivalents (for reference)

You can reproduce everything in this notebook from your terminal:

```bash
# Full historical re-ingest
python src/pipeline.py --mode full

# Incremental (only new days)
python src/pipeline.py --mode incremental

# Incremental for specific cities only
python src/pipeline.py --mode incremental --cities Baku Ganja

# Custom paths
python src/pipeline.py --mode full --db-path data/weather.duckdb --data-dir data --log-dir logs
```
